# NBA 3.1 — Pipeline Configuration

Single source of truth for all parameters, cohort logic, and helper functions
used across the patient journey (NB01–NB06) and HCP feature engineering notebooks.

**Location:** `/kerendia_nba_3_1_develop_clone/00_cohort_config`  
**Usage:** `%run "../00_cohort_config"` from any subfolder notebook.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, lit
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DateType
from dateutil.relativedelta import relativedelta
from datetime import date, datetime, timedelta
import calendar
import builtins
import sys

## Section 1 — Global Parameters
Shared across patient journey and HCP feature notebooks.

In [0]:
# ============================================================
# GLOBAL PARAMETERS — change these to configure the pipeline
# ============================================================

OFFSET_DAY       = 7              # Day-of-month offset (0 = standard calendar months)
DATA_START_DATE  = "2023-01-01"   # Exclude all data before this date
COHORT_ORIGIN    = "2025-01-01"   # Reference month of C1
COHORT_LAST      = "2026-10-01"   # Reference month of C22 (last cohort)
COHORT_MONTH     = "2025-02"      # Single HCP cohort (NB01; None = skip HCP window computation)
COHORT_MONTHS    = None           # Multi-cohort runs (NB02+): None = all cohorts COHORT_ORIGIN..COHORT_LAST
                                  #   with complete data; or a list, e.g. ["2025-01", "2025-02"]
VALIDATION_MODE  = True           # True = filter to 10 test HCPs; False = production (all HCPs)

print(f"OFFSET_DAY       = {OFFSET_DAY}")
print(f"DATA_START_DATE  = {DATA_START_DATE}")
print(f"COHORT_ORIGIN    = {COHORT_ORIGIN}")
print(f"COHORT_LAST      = {COHORT_LAST}")
print(f"COHORT_MONTH     = {COHORT_MONTH}")
print(f"COHORT_MONTHS    = {COHORT_MONTHS if COHORT_MONTHS is not None else 'all (complete data only)'}")
print(f"VALIDATION_MODE  = {VALIDATION_MODE}")

## Section 2 — Cohort Functions
`assign_cohort_month()` maps dates to offset-aligned months. Used by both patient journey and HCP features.

In [0]:
def assign_cohort_month(svc_dt_col, offset_days=None):
    """
    Maps a service-date column to its cohort reference month.

    With OFFSET_DAY = 0  → standard trunc to calendar month.
    With OFFSET_DAY = 7  → cohort boundaries fall on the 7th:
        C1 = [Jan 7, Feb 7),  C2 = [Feb 7, Mar 7), …

    The 'offset-th' day belongs to the NEW cohort (start-inclusive).
    offset_days=None uses the global OFFSET_DAY; pass a value (e.g. cb["offset_days"])
    to evaluate another offset, such as the offset-0 parity checks.
    """
    _off = OFFSET_DAY if offset_days is None else offset_days
    if _off == 0:
        return F.trunc(svc_dt_col, "month")
    else:
        return F.trunc(F.date_sub(svc_dt_col, _off - 1), "month")


def _cohort_month_sql(col_name):
    """
    Returns the SQL expression string for assign_cohort_month.
    Use inside F.expr() where a PySpark Column cannot be used (e.g. sequence()).
    """
    if OFFSET_DAY == 0:
        return f"trunc({col_name}, 'month')"
    else:
        return f"trunc(date_sub({col_name}, {OFFSET_DAY - 1}), 'month')"

In [0]:
# Build the cohort lookup: C1 through C22
_origin = date.fromisoformat(COHORT_ORIGIN)
_last   = date.fromisoformat(COHORT_LAST)

cohort_rows = []
_current = _origin
_idx = 1
while _current <= _last:
    _next = _current + relativedelta(months=1)
    if OFFSET_DAY == 0:
        cohort_start = _current
        cohort_end   = _next
    else:
        cohort_start = _current.replace(day=OFFSET_DAY)
        cohort_end   = _next.replace(day=OFFSET_DAY)
    cohort_rows.append((f"C{_idx}", _current, cohort_start, cohort_end))
    _current = _next
    _idx += 1

schema = StructType([
    StructField("cohort_label",      StringType(), False),
    StructField("cohort_ref_month",  DateType(),   False),
    StructField("cohort_start",      DateType(),   False),
    StructField("cohort_end",        DateType(),   False),
])

df_cohort_ref = spark.createDataFrame(cohort_rows, schema=schema)
print(f"{len(cohort_rows)} cohorts defined (C1 – C{len(cohort_rows)})")

## Section 3 — Validation / Production Filters
Controlled by `VALIDATION_MODE` in Section 1.  
* `get_test_npis()` — returns the 10 NPI list or `None` (data source)  
* `apply_hcp_filter(df)` — patient-level filter (for patient journey notebooks: finds patients linked to test NPIs)  
* `apply_npi_filter(df)` — NPI-level filter (for HCP feature notebooks: filters directly on NPI column)  

Both filter functions no-op in production mode.

In [0]:
# ============================================================
# VALIDATION / PRODUCTION FILTERS
# Controlled by VALIDATION_MODE in Section 1.
# ============================================================

def get_test_npis():
    """Return the 10 validation HCPs when in validation mode, None for production."""
    if not VALIDATION_MODE:
        return None
    return [
        "1396374062", "1508230566", "1801062104", "1386651560", "1528629052",
        "1659887388", "1215675616", "1467950774", "1235697863", "1841364841"
    ]

# Expose TEST_NPIS for backward compatibility (None in production mode)
TEST_NPIS = get_test_npis()

# HCP used by the HCP trace cell of each HCP feature notebook: the busiest test HCP of that
# notebook's source from DATA_START_DATE (checked 2026-10-23). Fixed so reruns trace the same HCP.
HCP_TRACE_NPIS = {
    "rx":     "1386651560",   # NB01, Rx_01_new: 18,938 claims
    "lab":    "1528629052",   # NB02, Lab_03: 64 claims
    "dx":     "1508230566",   # NB03, Dx_03: 1,662 claims
    "claims": "1386651560",   # NB04, Rx_01_new
    "nps":    "1386651560",   # NB05, Rx_05
}


# --- Patient-level filter (patient journey notebooks) ---
def apply_hcp_filter(df, npi_col="NPI_NUMBER", patient_col="PATIENT_ID"):
    """
    Filters a DataFrame to only patients linked to the test HCPs.
    Use in patient journey notebooks where data is at patient grain.
    In production mode (VALIDATION_MODE=False), returns the unfiltered df.
    Returns (filtered_df, patient_list).
    """
    npis = get_test_npis()
    if npis is None:
        print("Production mode — no patient filter applied")
        return df, None
    test_patients = [
        row[patient_col] for row in
        df.filter(F.col(npi_col).isin(npis))
          .select(patient_col).distinct().collect()
    ]
    df_filtered = df.filter(F.col(patient_col).isin(test_patients))
    print(f"Validation mode — filtered to {len(test_patients)} patients across {len(npis)} HCPs")
    return df_filtered, test_patients


# --- NPI-level filter (HCP feature notebooks) ---
def apply_npi_filter(df, npi_col="NPI_NUMBER"):
    """
    Filters a DataFrame directly on NPI_NUMBER.
    Use in HCP feature notebooks where data is at NPI grain.
    In production mode (VALIDATION_MODE=False), returns the unfiltered df.
    """
    npis = get_test_npis()
    if npis is None:
        print("Production mode — no NPI filter applied")
        return df
    df_filtered = df.filter(F.col(npi_col).isin(npis))
    print(f"Validation mode — filtered to {len(npis)} test NPIs")
    return df_filtered

## Section 4 — HCP Feature Engineering Helpers
Reusable functions for the claims monthly notebooks (01–06).

In [0]:
# ============================================================
# HCP FEATURE HELPERS — Reusable across all claims notebooks
# ============================================================

def clean_col_name(name):
    """Apply the same column-name cleaning as the original notebooks.
    Parentheses removal comes from 02_claim_lab_monthly (e.g. 'STAGE 2 (MILD)');
    no Rx values contain parentheses, so NB01 names are unchanged."""
    return (name
            .replace(", ", " ")
            .replace("(", "")
            .replace(")", "")
            .replace("+", "and")
            .replace("/", " or")
            .replace("-", "_")
            .replace(" ", "_")
            .replace("__", "_")
            .upper())


def pivoted_counts(df, npi_col, pivot_col, prefix, suffix, values=None):
    """
    Pivot df on pivot_col, count claims per NPI, clean column names.
    values=None: collect the distinct pivot values present in df (NB01 behaviour).
    values=list: use that fixed list, so every cohort/window gets the same columns
                 (values with no claims become 0).
    """
    if values is None:
        _vals = sorted(
            [r[0] for r in df.select(pivot_col).distinct().collect() if r[0] is not None]
        )
    else:
        _vals = sorted(values)
    df_piv = (
        df.groupBy(npi_col)
        .pivot(pivot_col, values=_vals)
        .agg(F.count(F.lit(1)).alias("cnt"))
        .na.fill(0)
    )
    new_names = [npi_col] + [
        f"{prefix}{clean_col_name(v)}{suffix}" for v in _vals
    ]
    return df_piv.toDF(*new_names)


def filter_to_window(df, svc_dt_col, w_start, w_end):
    """Filter DataFrame to a date-range window (exclusive start, inclusive end)."""
    return df.filter(
        (F.col(svc_dt_col) > F.lit(w_start)) &
        (F.col(svc_dt_col) <= F.lit(w_end))
    )


def build_feature_group(df_rx, npi_col, svc_dt_col, pivot_col, prefix,
                        window_bounds, extra_filter=None, combine_cols=None,
                        pivot_values=None):
    """
    Build one feature group across all lookback windows.
    Filters claims to each date range, optionally applies extra_filter,
    optionally creates a compound pivot column, then pivots and counts.
    """
    window_dfs = []
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        if extra_filter:
            df_w = df_w.filter(extra_filter)
        actual_pivot = pivot_col
        if combine_cols:
            df_w = df_w.withColumn(
                "_pivot_combined",
                F.concat_ws("_", *[F.col(c) for c in combine_cols])
            )
            actual_pivot = "_pivot_combined"
        suffix = f"_IN_LAST_{w}_MONTH"
        df_features = pivoted_counts(df_w, npi_col, actual_pivot, prefix, suffix,
                                     values=pivot_values)
        window_dfs.append(df_features)
    combined = window_dfs[0]
    for df_next in window_dfs[1:]:
        combined = combined.join(df_next, on=npi_col, how="outer")
    return combined.na.fill(0)


def build_total_col(df_rx, npi_col, svc_dt_col, window_bounds, total_name="TOTAL_RX_CLAIMS"):
    """Build {total_name}_IN_LAST_{N}_MONTH per NPI per window (denominators for PCT).
    total_name defaults to TOTAL_RX_CLAIMS (NB01); NB02 uses TOTAL_LAB_CLAIMS."""
    dfs = []
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        df_total = (
            df_w.groupBy(npi_col)
            .agg(F.count(F.lit(1)).alias(f"{total_name}_IN_LAST_{w}_MONTH"))
        )
        dfs.append(df_total)
    combined = dfs[0]
    for df_next in dfs[1:]:
        combined = combined.join(df_next, on=npi_col, how="outer")
    return combined.na.fill(0)

In [0]:
# ============================================================
# MATERIALIZE — compute an intermediate table once and reuse it
# ============================================================
# Serverless has no .cache(). Used by the fast cohort engines of the HCP feature notebooks.
#   Validation (10 NPIs, small): kept in memory as a local DataFrame.
#   Production: written to a staging Delta table <stage_prefix><name> and read back.
# stage_prefix: full path prefix of the staging tables. Default: the calling notebook's
# _STAGE_PREFIX (each notebook sets its own, e.g. .../notebook_output/claim_rx_monthly_cohort_stage_).

def materialize(df, name, stage_prefix=None):
    if VALIDATION_MODE:
        return spark.createDataFrame(df.toPandas(), schema=df.schema)
    prefix = stage_prefix if stage_prefix is not None else globals().get("_STAGE_PREFIX")
    if not prefix:
        raise ValueError("materialize: set _STAGE_PREFIX in the notebook or pass stage_prefix")
    path = prefix + name
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    return spark.read.format("delta").load(path)

## Section 5 — HCP Cohort Parameters
Lookback windows, cutoff date, and window bounds used by all HCP feature notebooks (01–06).  
Set `COHORT_MONTH` in Section 1. When `None`, this section is skipped (patient journey runs).

In [0]:
# ---- Alias for backward compat ----
OFFSET_DAYS = OFFSET_DAY

# ---- Column-name constants ----
NPI_COL = "NPI_NUMBER"
SVC_COL = "SVC_DT"

# ---- Lookback config ----
# 3.1 decision (2026-10-23): the same windows in every HCP notebook = the 3.0 windows. The 9-month
# window an earlier 3.1 version added to NB01/NB02 is dropped (not in 3.0, no model feature uses it).
LOOKBACK_WINDOWS = [1, 3, 6, 12]
AVG_EXTRA_MONTH  = True   # True = replicate old step3_lag_avg (1 month wider)

# ---- Window bounds (only when COHORT_MONTH is set) ----
def _build_bounds(lookback_windows, cutoff, cohort_start, extra_month=False, offset_days=None):
    """
    Build {window_size: {"start": date, "end": date}} for each lookback window.
    At offset 0 the exclusive start snaps to calendar-month boundaries.
    At offset > 0 the start is cutoff minus N calendar months.
    When extra_month is True (AVG behaviour), the window extends one month earlier.
    offset_days=None uses the global OFFSET_DAYS.
    """
    _off = OFFSET_DAYS if offset_days is None else offset_days
    bounds = {}
    for w in lookback_windows:
        end = cutoff
        if _off == 0:
            months_back = w - 1
            if extra_month:
                months_back += 1
            start_month = cohort_start - relativedelta(months=months_back)
            start = start_month - timedelta(days=1)
        else:
            months_back = w
            if extra_month:
                months_back += 1
            start = cutoff - relativedelta(months=months_back)
        bounds[w] = {
            "start": start.strftime("%Y-%m-%d"),
            "end":   end.strftime("%Y-%m-%d"),
        }
    return bounds

def _cohort_cutoff(cohort_start, off):
    """
    Last day (inclusive) of a cohort, consistent with assign_cohort_month():
    the offset-th day starts the NEW cohort.
      offset 0 -> month end                  (Feb cohort: up to Feb 28)
      offset 7 -> month end + (offset - 1)   (Feb cohort: up to Mar 6; Mar 7 starts March)
    """
    _month_end = datetime(cohort_start.year, cohort_start.month,
                          calendar.monthrange(cohort_start.year, cohort_start.month)[1])
    return _month_end + timedelta(days=(off - 1) if off > 0 else 0)


def get_cohort_bounds(cohort_month, offset_days=None):
    """
    All dates for one cohort month ('YYYY-MM'):
    cohort_id, cutoff (last day of the cohort, see _cohort_cutoff), window_bounds,
    avg_window_bounds. Same rules as the single-cohort globals below.
    """
    _off = OFFSET_DAYS if offset_days is None else offset_days
    _cs  = datetime.strptime(f"{cohort_month}-01", "%Y-%m-%d")
    _cut = _cohort_cutoff(_cs, _off)
    return {
        "cohort_month":      cohort_month,
        "cohort_id":         f"{cohort_month}_offset{_off}",
        "offset_days":       _off,
        "cutoff":            _cut,
        "window_bounds":     _build_bounds(LOOKBACK_WINDOWS, _cut, _cs, False, _off),
        "avg_window_bounds": _build_bounds(LOOKBACK_WINDOWS, _cut, _cs, AVG_EXTRA_MONTH, _off),
    }


def get_cohort_months(max_data_date=None, offset_days=None):
    """
    Cohort months to process: COHORT_MONTHS if set, else every month from
    COHORT_ORIGIN to COHORT_LAST. If max_data_date is given, cohorts whose cutoff
    is after it (incomplete most-recent window) are dropped.
    """
    if COHORT_MONTHS is not None:
        months = list(COHORT_MONTHS)
    else:
        months, _m = [], date.fromisoformat(COHORT_ORIGIN)
        while _m <= date.fromisoformat(COHORT_LAST):
            months.append(_m.strftime("%Y-%m"))
            _m += relativedelta(months=1)
    if max_data_date is not None:
        _max = datetime.combine(max_data_date, datetime.min.time()) if isinstance(max_data_date, date) and not isinstance(max_data_date, datetime) else max_data_date
        months = [m for m in months if get_cohort_bounds(m, offset_days)["cutoff"] <= _max]
    return months


# ---- One cohort list for every HCP feature notebook ----
# Each source ends on a different day (e.g. Lab_03/Lab_04 end 2026-08-31, Rx/Dx 2026-08-28). If each
# notebook used its own last date, a cohort could be complete for one table and missing in another.
# So all HCP notebooks use the EARLIEST last service date over their inputs. The sources are read on
# the first call only, so the patient-journey notebooks that %run this config don't pay for it.
HCP_SOURCE_TABLES = ["Rx_01_new", "Lab_03", "Lab_04", "Dx_03", "kerendia_nba_3_0_Rx_05"]
_HCP_DATA_MAX_DT  = None

def get_hcp_data_max_date():
    """Earliest last SVC_DT over HCP_SOURCE_TABLES (complete data in every HCP input)."""
    global _HCP_DATA_MAX_DT
    if _HCP_DATA_MAX_DT is None:
        # HCP_QUERY_OUTPUT is defined in the next cell ("HCP shared constants"); read at call time
        _ends = {t: spark.read.format("delta").load(HCP_QUERY_OUTPUT + t).agg(F.max(SVC_COL)).first()[0]
                 for t in HCP_SOURCE_TABLES}
        _HCP_DATA_MAX_DT = builtins.min(_ends.values())
        print("HCP sources end: " + ", ".join(f"{t} {d}" for t, d in _ends.items())
              + f" -> common end {_HCP_DATA_MAX_DT}")
    return _HCP_DATA_MAX_DT

def get_hcp_cohort_months(offset_days=None):
    """Cohorts written by every HCP feature notebook: get_cohort_months() with the common end date."""
    return get_cohort_months(max_data_date=get_hcp_data_max_date(), offset_days=offset_days)


if COHORT_MONTH is not None:
    _cohort_start = datetime.strptime(f"{COHORT_MONTH}-01", "%Y-%m-%d")
    CUTOFF_DATE   = _cohort_cutoff(_cohort_start, OFFSET_DAYS)
    COHORT_ID     = f"{COHORT_MONTH}_offset{OFFSET_DAYS}"

    WINDOW_BOUNDS     = _build_bounds(LOOKBACK_WINDOWS, CUTOFF_DATE, _cohort_start, extra_month=False)
    AVG_WINDOW_BOUNDS = _build_bounds(LOOKBACK_WINDOWS, CUTOFF_DATE, _cohort_start, extra_month=AVG_EXTRA_MONTH)

    print(f"Cohort: {COHORT_MONTH} | Offset: {OFFSET_DAYS} days | "
          f"Cutoff: {CUTOFF_DATE.strftime('%Y-%m-%d')} | ID: {COHORT_ID}")
    for w in LOOKBACK_WINDOWS:
        wb = WINDOW_BOUNDS[w]
        ab = AVG_WINDOW_BOUNDS[w]
        avg_months = w + (1 if AVG_EXTRA_MONTH else 0)
        print(f"  Last {w:2d}m: {wb['start']} -> {wb['end']}  "
              f"(AVG: {ab['start']} -> {ab['end']}, /{avg_months + 1})")
else:
    CUTOFF_DATE       = None
    COHORT_ID         = None
    WINDOW_BOUNDS     = None
    AVG_WINDOW_BOUNDS = None
    print("COHORT_MONTH is None — HCP window bounds skipped (patient journey mode)")

In [0]:
# ============================================================
# HCP SHARED CONSTANTS -- one place for every value the HCP feature notebooks (NB01-NB05) share.
# Feature notebooks must NOT redefine these (no hardcoded paths, brands, windows or table names).
# ============================================================

# ---- Paths ----
HCP_VOLUME          = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/"
                       "kerendia_nba_3_volume/hcp_level_model/ops_pipeline/")
HCP_QUERY_OUTPUT    = HCP_VOLUME + "query_output/"      # inputs (00_Write_Input_Files, patient journey)
HCP_NOTEBOOK_OUTPUT = HCP_VOLUME + "notebook_output/"   # feature outputs, 3.0 originals, staging tables

# ---- Brand list (the original 3.0 brand filter, used by NB01, NB04, NB05) ----
HCP_BRANDS = ["INVOKANA 100MG", "INVOKANA 300MG", "FARXIGA", "JARDIANCE",
              "SPIRONOLACTONE-MRA", "Diuretic + MRA", "DPP-IV", "GLP-1"]
HCP_BRAND_FILTER = "Category IN (" + ", ".join(f"'{b}'" for b in HCP_BRANDS) + ")"   # SQL form (NB01)

# ---- History start of the Rx_05 input (original NB05 filter) ----
# Not a feature start: NB05 uses the history from here for first-start dates only. Counted claims
# start at DATA_START_DATE, as in every notebook.
HCP_RX_HISTORY_START = "2019-11-01"

# ---- Lookback windows per notebook: all the same (LOOKBACK_WINDOWS = 1, 3, 6, 12) ----
# NB01/NB02 use LOOKBACK_WINDOWS directly; NB03-NB05 read their entry here.
HCP_LOOKBACK_WINDOWS = {k: list(LOOKBACK_WINDOWS) for k in ["rx", "lab", "dx", "claims", "nps"]}

# ---- Patient-journey tables (in query_output/): (validation table, production table) ----
# Validation mode reads the 3.1 patient-journey validation outputs; production still reads 3.0
# (calendar months) until a 3.1 production output exists.
HCP_PATIENT_TABLES = {
    "new_and_total_treatments": ("kerendia_nba_3_1_pat_new_and_total_treatments_validation",
                                 "pat_new_and_total_treatments"),                    # NB01
    "new_ckd_stage":            ("kerendia_nba_3_1_pat_new_ckd_stage_validation",
                                 "pat_new_ckd_stage"),                               # NB03
    "gfr_dx_stage":             ("kerendia_nba_3_1_pat_gfr_dx_stage_validation",
                                 "pat_gfr_dx_stage_and_treatment_category"),         # NB04 family A
    "uacr_stage":               ("kerendia_nba_3_1_pat_uacr_stage_validation",
                                 "pat_uacr_stage_and_treatment_category_new"),       # NB04 family C
}

def get_patient_table(key):
    """Name of the patient-journey table for this mode (VALIDATION_MODE)."""
    _val, _prod = HCP_PATIENT_TABLES[key]
    return _val if VALIDATION_MODE else _prod

def get_patient_table_30(key):
    """3.0 table (calendar months): the input of the offset-0 parity checks."""
    return HCP_PATIENT_TABLES[key][1]

# ---- 3.0 provider mapping (NPI -> PROVIDER_ID) ----
HCP_NPI_MAPPING_TABLE = "kerendia_nba_3_0_npi_bhid_mapping"

print(f"HCP constants: {len(HCP_BRANDS)} brands | windows {HCP_LOOKBACK_WINDOWS} | "
      f"patient tables: {'validation' if VALIDATION_MODE else 'production (3.0)'}")

In [0]:
# ---- All-cohorts engine for feature-row tables (pattern A for NB04 / NB05) ----
# Input: a long feature-row table (key, SVC_DT, FEATURE, N), as built by NB04/NB05 build_feature_rows.
# 1) rows -> (key, cohort month, FEATURE, N): one aggregation per run.
# 2) every cohort month is tagged with each (cohort, window) it belongs to through a small broadcast
#    table, then ONE pivot by key x cohort. Windows line up with cohort months (checked below), so
#    counting by cohort month gives the same values as filtering by date.
from datetime import datetime as _dt, date as _date, timedelta as _td

_HCP_EARLIEST = _date(1900, 1, 1)

def hcp_cm_of(d, off):
    """Python version of assign_cohort_month (datetime in, datetime out)."""
    return (d - _td(days=off - 1)).replace(day=1) if off > 0 else d.replace(day=1)

def hcp_window_months(b, off):
    """Window bounds {start (exclusive), end (inclusive)} -> (first, last) cohort month, as dates.
    Fails if the window does not start and end on cohort-month boundaries."""
    s = _dt.strptime(b["start"], "%Y-%m-%d")
    e = _dt.strptime(b["end"], "%Y-%m-%d")
    lo, hi = hcp_cm_of(s + _td(days=1), off), hcp_cm_of(e, off)
    if hcp_cm_of(s, off) == lo or hcp_cm_of(e + _td(days=1), off) == hi:
        raise ValueError(f"Window {b} does not line up with cohort months at offset {off}")
    return lo.date(), hi.date()

def _hcp_single_offset(cbs):
    offs = {cb["offset_days"] for cb in cbs}
    if len(offs) != 1:
        raise ValueError(f"All cohorts must share one offset, got {offs}")
    return offs.pop()

def feature_rows_to_months(rows, key_col, offset_days, date_col=None):
    """(key, date, FEATURE, N) -> (key, _CM, FEATURE, N). Rows with N = 0 are kept, so a key whose
    rows only carry column names (NB04 family B) still gets an output row, as in the per-cohort code."""
    date_col = date_col or SVC_COL
    return (rows.withColumn("_CM", assign_cohort_month(F.col(date_col), offset_days))
                .groupBy(key_col, "_CM", "FEATURE").agg(F.sum("N").alias("N")))

def build_all_cohorts_from_months(monthly, cbs, key_col, feature_values, windows):
    """One row per key x cohort: key_col, COHORT_ID, then {v}_IN_LAST_{w}_MONTH (for w, for v) and
    RUNNING_{v}, counts filled with 0. Keys = keys with any row up to the cohort (= RUNNING_ universe).
    `monthly` must be built at the cohorts' offset (feature_rows_to_months)."""
    off = _hcp_single_offset(cbs)
    tag_rows = []
    for cb in cbs:
        cm = _dt.strptime(f"{cb['cohort_month']}-01", "%Y-%m-%d").date()
        for w in windows:
            lo, hi = hcp_window_months(cb["window_bounds"][w], off)
            if hi != cm:
                raise ValueError(f"{cb['cohort_id']}: window {w} ends in {hi}, expected {cm}")
            tag_rows.append((cb["cohort_id"], f"_IN_LAST_{w}_MONTH", lo, cm))
        tag_rows.append((cb["cohort_id"], "RUNNING", _HCP_EARLIEST, cm))   # RUNNING_: all months up to the cohort
    tags = F.broadcast(spark.createDataFrame(tag_rows, "COHORT_ID string, _TAG string, _LO date, _HI date"))

    window_cols  = [f"{v}_IN_LAST_{w}_MONTH" for w in windows for v in feature_values]
    running_cols = [f"RUNNING_{v}" for v in feature_values]
    return (monthly
            .join(tags, (F.col("_CM") >= F.col("_LO")) & (F.col("_CM") <= F.col("_HI")))
            .withColumn("_COL", F.when(F.col("_TAG") == "RUNNING", F.concat(F.lit("RUNNING_"), F.col("FEATURE")))
                                 .otherwise(F.concat(F.col("FEATURE"), F.col("_TAG"))))
            .groupBy(key_col, "COHORT_ID").pivot("_COL", window_cols + running_cols).agg(F.sum("N"))
            .na.fill(0, subset=window_cols + running_cols))

def add_cohort_meta(df, cbs):
    """Adds COHORT_MONTH, OFFSET_DAYS, CUTOFF_DATE, SVC_DT_M, PULL_DT by COHORT_ID (same types as the
    per-cohort code: string, int, string, timestamp, date)."""
    meta = (spark.createDataFrame(
                [(cb["cohort_id"], cb["cohort_month"], int(cb["offset_days"]), cb["cutoff"].strftime("%Y-%m-%d"))
                 for cb in cbs],
                "COHORT_ID string, COHORT_MONTH string, OFFSET_DAYS int, CUTOFF_DATE string")
            .withColumns({"SVC_DT_M": F.to_timestamp(F.concat(F.col("COHORT_MONTH"), F.lit("-01"))),
                          "PULL_DT": F.current_date()}))
    return df.join(F.broadcast(meta), on="COHORT_ID", how="inner")

In [0]:
# Quick self-test: verify assign_cohort_month with known examples
from datetime import date as _d

_test_cases = [
    ("2025-01-01", 0,  "2025-01-01"),
    ("2025-01-07", 7,  "2025-01-01"),
    ("2025-01-06", 7,  "2024-12-01"),
    ("2025-02-06", 7,  "2025-01-01"),
    ("2025-02-07", 7,  "2025-02-01"),
    ("2025-10-04", 7,  "2025-09-01"),
]

print(f"Testing with current OFFSET_DAY = {OFFSET_DAY}")
print("-" * 55)
for svc, off, expected in _test_cases:
    if off != OFFSET_DAY:
        continue
    test_df = spark.createDataFrame([(svc,)], ["SVC_DT"]).withColumn("SVC_DT", F.to_date("SVC_DT"))
    result = test_df.select(assign_cohort_month(col("SVC_DT")).alias("cohort_month")).collect()[0][0]
    status = "PASS" if str(result) == expected else "FAIL"
    print(f"  {svc} → {result}  (expected {expected})  [{status}]")

# HCP window boundary must agree with assign_cohort_month(): every day in a cohort's
# 1-month window maps to that cohort, and the days just outside map to its neighbours.
# Pure Python (same rule as assign_cohort_month), checked for every cohort.
def _py_cohort_month(d, off):
    return (d - timedelta(days=off - 1)).replace(day=1) if off > 0 else d.replace(day=1)

_bad = []
for _m in get_cohort_months():
    _cb  = get_cohort_bounds(_m)
    _s   = datetime.strptime(_cb["window_bounds"][1]["start"], "%Y-%m-%d")   # exclusive
    _e   = datetime.strptime(_cb["window_bounds"][1]["end"], "%Y-%m-%d")     # inclusive
    _ref = datetime.strptime(f"{_m}-01", "%Y-%m-%d")
    if not (_py_cohort_month(_s + timedelta(days=1), OFFSET_DAY) == _ref    # first day in
            and _py_cohort_month(_e, OFFSET_DAY) == _ref                    # last day in
            and _py_cohort_month(_s, OFFSET_DAY) != _ref                    # day before out
            and _py_cohort_month(_e + timedelta(days=1), OFFSET_DAY) != _ref):  # day after out
        _bad.append(_m)
_ex = get_cohort_bounds("2025-02")
print(f"\nHCP window vs cohort assignment ({len(get_cohort_months())} cohorts): "
      f"{'PASS' if not _bad else 'FAIL ' + str(_bad)}")
print(f"  e.g. 2025-02: 1-month window ({_ex['window_bounds'][1]['start']}, {_ex['window_bounds'][1]['end']}], "
      f"cutoff {_ex['cutoff']:%Y-%m-%d}")

print("\n✅ Config loaded. Available to all pipeline notebooks.")